# 7.6 · 多目标与帕累托 / Multi-objective & Pareto

> **课程定位 / Where this fits**
> 第 6 课，**Part 7 · 模型评估与优化**。
> Lesson 6, **Part 7 · Model Evaluation & Tuning**.
>
> 现实中很少只优化一个目标。上线模型常要同时权衡：**精度 vs 推理速度**（APP 要快）、**精度 vs 模型大小**（端侧设备）、**精度 vs 公平性**(7.10)。当目标相互冲突时，没有"唯一最优"，只有**一组互不被支配的折中方案**——这就是**帕累托前沿**。本课讲怎么找到它、怎么从中选。
> Real problems rarely optimize one objective. Production models trade off **accuracy vs inference speed** (apps need speed), **accuracy vs model size** (edge devices), **accuracy vs fairness** (7.10). When objectives conflict there's no single optimum, only a **set of non-dominated trade-offs** — the **Pareto frontier**. This lesson finds it and picks from it.
>
> 💼 **实战/面试视角**："精度和延迟怎么权衡 / 怎么选模型上线" 是系统设计/MLOps 常考。
> 💼 **Practical/interview angle:** "trading accuracy vs latency / choosing a model to ship" — system-design/MLOps.

> 💡 **面试相关 / Interview-relevant**
> - "什么是帕累托最优 / 支配关系"（出镜率 ★★★）
> - "精度和速度冲突时怎么选模型"（★★★★）
> - "多目标怎么转成单目标（加权/约束）"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解**支配(dominance)** 与**帕累托前沿**。
   Understand dominance and the Pareto frontier.
2. 在"精度 vs 延迟"上算出帕累托前沿。
   Compute the Pareto frontier for accuracy vs latency.
3. 学会从前沿上**按业务约束选点**。
   Pick a point from the frontier by business constraints.
4. 了解把多目标转成单目标的两种常用法（加权、约束）。
   Know the two common scalarizations (weighting, constraint).

## 目录 / TOC
1. [先建直觉：支配与帕累托前沿 ⭐](#1)
2. [📊 精度 vs 延迟：训练候选模型](#2)
3. [计算帕累托前沿 ⭐](#3)
4. [从前沿选点 + 标量化 ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 先建直觉：支配与帕累托前沿 ⭐ / Intuition: Dominance & the Frontier

假设我们要同时**最大化精度、最小化延迟**。比较两个模型 A 和 B：
Suppose we want to **maximize accuracy and minimize latency** at once. Compare models A and B:
- 如果 A **在两个目标上都不比 B 差，且至少一个更好**，就说 **A 支配 B**——B 没有任何理由被选（A 全面更优）。
  If A is **no worse on both objectives and strictly better on at least one**, then **A dominates B** — there's no reason to pick B.
- **帕累托前沿**：所有**不被任何其它方案支配**的方案的集合。前沿上的点彼此"各有所长"——想更准就得牺牲速度，想更快就得牺牲精度，**无法同时改进**。
  **Pareto frontier:** the set of solutions **not dominated by any other**. Points on it each have a strength — to gain accuracy you sacrifice speed and vice versa; you **can't improve both**.

实战意义：**只该从帕累托前沿上选模型**——前沿之外的模型都被前沿上的某个点全面碾压，选它们是浪费。具体选前沿上哪一点，由业务约束决定（如"延迟必须 <10ms"）。
Practical meaning: **only ever pick from the Pareto frontier** — anything off it is strictly beaten by a frontier point, so picking it wastes performance. Which frontier point depends on business constraints (e.g. "latency must be <10ms").


<a id="2"></a>
## 2. 精度 vs 延迟：训练候选模型 / Training Candidates

造一批复杂度不同的模型（不同 `n_estimators` 的随机森林 + 逻辑回归 + 单树），对每个测量**两个目标**：测试精度（越高越好）和单次推理延迟（越低越好）。这模拟了真实的"选哪个模型上线"决策。
We make models of varying complexity (random forests with different `n_estimators` + logistic regression + a single tree), and measure **two objectives** for each: test accuracy (higher better) and inference latency (lower better). This mirrors a real "which model to ship" decision.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
sns.set_theme(style="whitegrid")

X, y = make_classification(n_samples=6000, n_features=20, n_informative=10, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

candidates = {
    "LogReg": LogisticRegression(max_iter=1000),
    "Tree(d5)": DecisionTreeClassifier(max_depth=5, random_state=0),
    "RF-10": RandomForestClassifier(n_estimators=10, random_state=0, n_jobs=1),
    "RF-50": RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=1),
    "RF-200": RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=1),
    "RF-500": RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=1),
}
results = []
for name, m in candidates.items():
    m.fit(X_tr, y_tr)
    acc = m.score(X_te, y_te)                                  # 目标1: 精度(越高越好)
    t = time.perf_counter()
    for _ in range(5): m.predict(X_te)                         # 重复几次测推理延迟
    latency = (time.perf_counter() - t) / 5 * 1000            # 目标2: 延迟ms(越低越好)
    results.append({"name": name, "accuracy": acc, "latency_ms": latency})
    print(f"{name:<10} 精度 accuracy={acc:.4f}  延迟 latency={latency:6.2f} ms")


<a id="3"></a>
## 3. 计算帕累托前沿 ⭐ / Computing the Pareto Frontier

判定一个点是否在前沿上：检查**有没有任何其它点支配它**（精度 ≥ 它且延迟 ≤ 它，且至少一项严格更好）。没有被支配的就在前沿上。
To test if a point is on the frontier: check whether **any other point dominates it** (accuracy ≥ and latency ≤, with at least one strict). If none does, it's on the frontier.


In [ ]:
res = results
def is_dominated(p, others):
    # p 被某个 q 支配: q 精度≥p 且 q 延迟≤p, 且至少一项严格更好 / does any q dominate p?
    for q in others:
        if q is p: continue
        if (q["accuracy"] >= p["accuracy"] and q["latency_ms"] <= p["latency_ms"]
                and (q["accuracy"] > p["accuracy"] or q["latency_ms"] < p["latency_ms"])):
            return True
    return False

for p in res:
    p["on_frontier"] = not is_dominated(p, res)        # 不被支配 = 在帕累托前沿上

frontier = sorted([p for p in res if p["on_frontier"]], key=lambda d: d["latency_ms"])
print("帕累托前沿上的模型(无法被全面超越):")
for p in frontier:
    print(f"  {p['name']:<10} 精度={p['accuracy']:.4f}  延迟={p['latency_ms']:.2f} ms")
print("\n被支配的模型(有更优选择, 不该上线):")
for p in res:
    if not p["on_frontier"]:
        print(f"  {p['name']:<10} 精度={p['accuracy']:.4f}  延迟={p['latency_ms']:.2f} ms")

fig, ax = plt.subplots(figsize=(7.5, 5))
for p in res:
    c = "C2" if p["on_frontier"] else "C3"
    ax.scatter(p["latency_ms"], p["accuracy"], s=110, color=c, zorder=3)
    ax.annotate(p["name"], (p["latency_ms"], p["accuracy"]), xytext=(5,5), textcoords="offset points", fontsize=9)
ax.plot([p["latency_ms"] for p in frontier], [p["accuracy"] for p in frontier], "g--", alpha=0.6, label="帕累托前沿 frontier")
ax.set_xlabel("延迟 latency (ms) — 越低越好"); ax.set_ylabel("精度 accuracy — 越高越好")
ax.legend(); ax.set_title("精度 vs 延迟: 绿=帕累托前沿(可选), 红=被支配(不该选)")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 从前沿选点 + 标量化 ⭐ / Picking from the Frontier & Scalarization

前沿给了你"所有合理候选"，但最终要选一个。两种常见做法：
The frontier gives "all reasonable candidates"; you still pick one. Two common approaches:
- **约束法(最常用，实战首选)**：定一个硬约束（如"延迟必须 ≤ X ms"），在满足约束的前沿点里选精度最高的。
  **Constraint (most common, practical default):** set a hard constraint (e.g. "latency ≤ X ms") and pick the highest-accuracy frontier point that satisfies it.
- **加权标量化**：把多目标加权成一个分数 $\text{score} = w_1\cdot\text{精度} - w_2\cdot\text{延迟}$，转成单目标优化。简单，但权重难定、且只能找到前沿的凸部分。
  **Weighted scalarization:** combine objectives into one score $w_1\cdot\text{acc} - w_2\cdot\text{latency}$, reducing to single-objective. Simple, but weights are arbitrary and it only finds the convex part of the frontier.


In [ ]:
# 约束法: 延迟必须 ≤ 30ms, 在满足的前沿点里选最准的 / pick under a latency budget
budget = 30.0
feasible = [p for p in frontier if p["latency_ms"] <= budget]
best = max(feasible, key=lambda d: d["accuracy"]) if feasible else None
print(f"约束法(延迟≤{budget}ms): 选 {best['name']} (精度={best['accuracy']:.4f}, 延迟={best['latency_ms']:.2f}ms)")

# 加权标量化: 把两目标合成一个分数 / weighted scalarization
# 先把延迟归一化到 [0,1] 再加权, 否则量纲不可比 / normalize before weighting
lat = np.array([p["latency_ms"] for p in res]); lat_norm = (lat - lat.min())/(lat.max()-lat.min())
for w_lat in [0.1, 0.5, 0.9]:                          # 延迟的权重(越大越在意速度)
    scores = [p["accuracy"] - w_lat*ln for p, ln in zip(res, lat_norm)]
    pick = res[int(np.argmax(scores))]
    print(f"加权(延迟权重={w_lat}): 选 {pick['name']:<8} → 越在意速度, 越倾向轻量模型")


<a id="5"></a>
## 5. 小结 / Summary

```
多目标: 目标冲突时(精度vs延迟/大小/公平), 没有唯一最优
支配: A 在所有目标≥B 且至少一个严格更优 → A 支配 B(B 不该被选)
帕累托前沿: 所有不被支配的方案; 只该从前沿上选模型(前沿外都被碾压)
选点: 约束法(定硬约束如延迟≤X, 选满足约束里最优的, 实战首选) / 加权标量化(合成单目标)
加权标量化注意: 不同量纲先归一化; 只能找凸前沿; 权重主观
```

### 💡 面试速查 / Interview cheat-sheet
1. **帕累托最优 = 不被任何方案支配**; 只从前沿选模型。
   Pareto-optimal = dominated by none; only pick from the frontier.
2. **支配**: 所有目标不差且至少一个更好。
   Dominance: no worse on all, strictly better on at least one.
3. **选点常用约束法**(延迟≤X 下选最准), 比加权直观可靠。
   Picking via a constraint (best accuracy under latency≤X) is more reliable than weighting.
4. **加权标量化要先归一化**, 且只能找凸前沿。
   Weighted scalarization needs normalization and finds only the convex frontier.
5. 实战常见权衡: **精度 vs 延迟/模型大小/公平性(7.10)**。
   Common trade-offs: accuracy vs latency/size/fairness.

### 下一节 / Next
**7.7 特征选择**——更少的特征常意味着更快、更可解释、更不易过拟合。filter/wrapper/embedded 三大类方法 + RFE。
**7.7 Feature Selection** — fewer features often means faster, more interpretable, less overfit. The three families (filter/wrapper/embedded) + RFE.
